# Bubble analysis

Reads the reviewed bubbles exported by `bubble_inference.ipynb` (`results/<RUN_NAME>/bubbles.csv`, `frames.csv`,
plus the outlines in `shapes/`). This notebook is a starting point: it loads the data, gives a per-train overview and
shows how to get pixel masks. Specific analyses get added below.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import bubble_io as bio

plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "0.9", "grid.linewidth": 0.6})
BLUE = "#2a78d6"   # single series colour

## CONFIG

In [ ]:
RUN_DIR = "results/r563_final"
REVIEWED_ONLY = True        # only frames ticked "Reviewed" in napari
IN_ANALYSIS_ONLY = True     # apply the exclusions chosen at export (inner_margin, drop_edge_bubbles, drop_unreliable_fits)
TRAIN_TIMES = None          # optional {train: time} for the time axis, e.g. {0: 0, 1: 3.5, ...} (minutes); None = train number
TIME_LABEL = "train"        # axis label, e.g. "time [min]"

## 1. Load

In [ ]:
bubbles, frames = bio.load_results(RUN_DIR, reviewed_only=REVIEWED_ONLY, in_analysis_only=IN_ANALYSIS_ONLY)
print(f"{len(frames)} frames from trains {sorted(frames['train'].unique().tolist())}, {len(bubbles)} bubbles")
tvals = (lambda t: TRAIN_TIMES[t]) if TRAIN_TIMES else (lambda t: t)
bubbles["t"] = bubbles["train"].map(tvals)
frames["t"] = frames["train"].map(tvals)
display(frames)
display(bubbles.head())

**Columns in `bubbles`** (lengths in native px, `_um` / `_um2` in µm):

| column | meaning |
|---|---|
| `frame`, `train`, `frame_idx`, `bubble_id` | where the bubble is from |
| `x`, `y` | centre (for edge bubbles the fitted centre, which may lie outside the image) |
| `area_px2`, `r_eq` | area and equivalent radius √(area/π) of the **full** bubble |
| `major_axis`, `minor_axis`, `orientation_deg`, `eccentricity` | ellipse description |
| `area_visible_px2` | part of the bubble inside the image |
| `edge_truncated`, `fit_kind`, `arc_deg`, `fit_reliable` | edge handling (`fit_kind`: mask / circle / ellipse / drawn_ellipse) |
| `in_analysis` | False if excluded by the export settings |
| `shape_type` | polygon or ellipse (how the outline is stored) |

## 2. Overview per train

If a train has several frames, values are the mean over its frames (± standard deviation in the plots).

In [ ]:
per_frame = bubbles.groupby(["train", "t", "frame"]).agg(
    n_bubbles=("r_eq_um", "size"),
    r_mean_um=("r_eq_um", "mean"),
    r_median_um=("r_eq_um", "median"),
    total_area_um2=("area_um2", "sum"),
).reset_index()
per_train = per_frame.groupby(["train", "t"]).agg(
    n_frames=("frame", "size"),
    n_bubbles=("n_bubbles", "mean"), n_bubbles_std=("n_bubbles", "std"),
    r_mean_um=("r_mean_um", "mean"), r_median_um=("r_median_um", "mean"),
    total_area_um2=("total_area_um2", "mean"),
).reset_index()
display(per_train.round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# bubble count per train
ax[0].errorbar(per_train["t"], per_train["n_bubbles"], yerr=per_train["n_bubbles_std"].fillna(0),
               color=BLUE, lw=2, marker="o", ms=8, capsize=3)
ax[0].set_xlabel(TIME_LABEL); ax[0].set_ylabel("bubbles per frame"); ax[0].set_title("Bubble count")
ax[0].set_ylim(bottom=0)

# equivalent-radius distribution per train
groups = [g["r_eq_um"].to_numpy() for _, g in bubbles.groupby("t")]
pos = sorted(bubbles["t"].unique())
width = 0.6 * (np.min(np.diff(pos)) if len(pos) > 1 else 1)
ax[1].boxplot(groups, positions=pos, widths=width, patch_artist=True, showfliers=False,
              boxprops=dict(facecolor=BLUE, alpha=0.35, edgecolor=BLUE), medianprops=dict(color=BLUE, lw=2),
              whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE))
ax[1].set_xlabel(TIME_LABEL); ax[1].set_ylabel("equivalent radius [µm]"); ax[1].set_title("Size distribution (box: quartiles)")
plt.tight_layout()

## 3. Look at a frame

In [ ]:
FRAME = frames["frame"].iloc[0]
img, shapes = bio.load_frame(RUN_DIR, FRAME)
bio.plot_shapes(img, shapes, title=f"{FRAME}: {len(shapes)} bubbles");

## 4. Pixel masks (when an analysis needs them)

`bio.shapes_to_masks(shapes, image_shape, scale)` gives one boolean mask per bubble (masks may overlap), at any
resolution (`scale=3` → 3× upsampled for sub-pixel accuracy). The example computes the fraction of the frame covered
by bubbles.

In [ ]:
masks = bio.shapes_to_masks(shapes, img.shape, scale=1)
covered = masks.any(axis=0)
print(f"{FRAME}: {len(masks)} masks of shape {masks.shape[1:]}, {100 * covered.mean():.1f}% of the frame covered by bubbles")

## 5. Analysis

*(to be added)*